# Introducing Essential Functional Concepts

In [12]:
from collections.abc import Generator
from typing import Any, Final

import ipytest
import pytest

ipytest.autoconfig()

* We'll look at each of the following functional programming topics:
    * First-class and higher-order functions (pure functions)
    * Immutable data
    * Strict and non-strict evaluation (eager versus lazy evaluation)
    * Recursion instead of an explicit loop state
    * Functional type systems
* Purely functional programming avoids the complexities of an explicit state maintained through variable assignments.

## First-class functions

* Functional programming is often succinct and expressive.
* One way to achieve it is by providing functions as arguments and return values for other functions.
* In Python, functions are first-class objects and can be manipulated just like all other objects.

## Pure functions

* To be expressive, a function should be free from the confusion created by **side effects**.
* Using pure functions can also allow some optimizations by changing **evaluation order**.
* Pure functions are conceptually simpler and much easier to **test**.
* To write a pure function in Python, we have to write local-only code.
* A Python lambda is a pure function.
    * Can't have assignment statements.

## Higher-order functions

* Higher-order functions are functions that
    * accept a function as an argument or
    * return a function as a value.
* We can use higher-order functions as a way to create composite functions from simpler functions.
* Consider the Python `max()` function, where we can provide a function as an argument and modify how the `max()` function behaves.

In [13]:
%%ipytest


def test_max_higher_order_function() -> None:
    year_cheese = ((2000, 29.87), (2007, 33.5), (2009, 33.02), (2010, 32.92))

    assert max(year_cheese) == (2010, 32.92)
    assert max(year_cheese, key=lambda yc: yc[1]) == (2007, 33.5)

.                                                                                            [100%]
1 passed in 0.01s


## Immutable data

* **Reliance on Immutable Objects:**
    * Functional programming favours approaches that do not rely on variables to track the state of a computation, encouraging the use of immutable objects.
    * `tuple`s and `namedtuple`s can be used extensively to construct more complex immutable data structures
    * Immutable `tuple`s can offer performance advantages over more complex mutable objects. In some cases, these benefits arise from redesigning algorithms to avoid the costs of object mutation.
* **Minimising Class Definitions:**
    * Class definitions are largely avoided because functional programming does not require stateful objects, although this does not mean eliminating objects altogether.
    * *Callable objects* can provide namespaces for related functions, support configurability, and facilitate caching for performance optimisation.
    * A callable object is an object that can be called like a function.
        * An instance of a class becomes callable when the class implements the special `__call__` method.
    * These are design options rather than requirements of functional programming.
    * Whether a callable object is appropriate depends on the problem being solved and the alternatives available.

In [14]:
%%ipytest


class TaxCalculator:
    def __init__(self, default_rate: float) -> None:
        # Configurability: retain the rate for subsequent calls.
        # Configuration does not need to be passed explicitly on every subsequent call.
        self.default_rate = default_rate

        # Caching: retain previously calculated results
        self.cache: dict[float, float] = {}

    def __call__(self, amount: float) -> float:
        # Caching: return a cached result if one exists
        if amount in self.cache:
            return self.cache[amount]

        result = amount * (1 + self.default_rate)
        self.cache[amount] = result

        return result

    def get_rate_percentage(self) -> str:
        # Namespace: related functionality within the same class
        return f"{self.default_rate * 100}%"


def test_tax_calculator_namespace_and_configurability() -> None:
    calculator = TaxCalculator(default_rate=0.20)

    assert calculator(100.0) == pytest.approx(120.0)
    assert calculator(150.0) == pytest.approx(180.0)

    assert calculator.get_rate_percentage() == "20.0%"


def test_tax_calculator_caching() -> None:
    calculator = TaxCalculator(default_rate=0.50)

    # First call calculates and caches the result.
    assert calculator(100.0) == pytest.approx(150.0)

    # Replace the cached value with a distinctive result.
    calculator.cache[100.0] = 999.0

    # A cache hit should return 999.0, not recalculate 120.0.
    assert calculator(100.0) == 999.0


..                                                                                           [100%]
2 passed in 0.01s


* **The Wrap-Process-Unwrap Pattern:**
    * Data structures, such as `list`s of `tuple`s, can be transformed into forms that simplify subsequent operations without modifying the original data.
        * `unwrap(process(wrap(structure)))`
    * Example: `max(map(lambda yc: (yc[1], yc), year_cheese))[1]`
        * **Wrapping:** Each item can be *wrapped* in a `tuple` containing a comparison key and the original item.
        * **Processing:** A higher-order function, such as `max()`, can then *process* these `tuple`s using their default comparison behaviour.
        * **Unwrapping the Result:** The original item can be extracted from the result using `tuple` indexing, such as `[1]`, or a helper function such as `snd()` (second).
            * In this case, `snd()` is just simply `snd = lambda x: x[1]`
            * `snd(max(map(lambda yc: (yc[1], yc), year_cheese)))`


In [15]:
%%ipytest


def test_wrap_process_unwrap() -> None:
    year_cheese = ((2000, 29.87), (2007, 33.5), (2009, 33.02), (2010, 32.92))
    max_result = max(map(lambda yc: (yc[1], yc), year_cheese))[1]

    assert max_result == (2007, 33.5)

    # Using an unwrap function and a generator expression instead of map()
    snd = lambda x: x[1]
    max_result_gen = snd(max((yc[1], yc) for yc in year_cheese))

    assert max_result_gen == (2007, 33.5)


.                                                                                            [100%]
1 passed in 0.01s


## Strict and non-strict evaluation

* Functional programming's efficiency stems, in part, from being able to defer a computation until it's required.
    * The idea of *lazy* or *non-strict* evaluation is very helpful.
* Logical expression operators `and`, `or`, and `if-then-else` are all non-strict.
    * Also known as *short-circuit* operators because they don't need to evaluate all arguments to determine the resulting value.
* Other parts of Python are *strict*.
    * Outside the logical operators, an expression is evaluated *eagerly* from left to right.
* Python's *generator expressions* and *generator functions* are *lazy*.
    * They don't create all possible results immediately.
    * They can be a little awkward for simple functional programming.
      * A generator can only be used once in Python.

In [16]:
%%ipytest


def eager_sequence(n: int) -> tuple[tuple[int, ...], list[int]]:
    trace = list(range(n))
    results = tuple(range(n))

    return results, trace


def lazy_sequence(n: int) -> tuple[Generator[int], list[int]]:
    trace: list[int] = []

    def generator() -> Generator[int]:
        for i in range(n):
            trace.append(i)
            yield i

    # If `generator()` executes eagerly, `trace` will be populated up to `n` immediately
    return generator(), trace


@pytest.mark.parametrize("sequence_func", [eager_sequence, lazy_sequence])
def test_eager_and_lazy(sequence_func) -> None:
    n: Final[int] = 5
    result_stream, trace = sequence_func(n)

    # Before consumption
    if sequence_func == eager_sequence:
        # Eager: The trace is already fully populated at call time
        assert len(trace) == n
    else:
        # Lazy: The trace is completely empty because execution hasn't started yet
        assert len(trace) == 0

    # Consume
    result_iter = iter(result_stream)
    assert next(result_iter) == 0

    # After consumption
    if sequence_func == eager_sequence:
        # Eager trace was already complete from the start
        assert len(trace) == n
    else:
        # Lazy trace now has exactly 1 item, proving it evaluated on demand
        assert len(trace) == 1

    assert trace[0] == 0


..                                                                                           [100%]
2 passed in 0.01s


## Recursion instead of an explicit loop state

* Avoidance of loop states:
    * Functional programs avoid the overhead of tracking explicit loop states by relying instead on the much simpler approach of recursive functions.
* Structure of recursion:
    * When defining a recursive function over a range, the base case handles an empty range, while a non-empty range is processed recursively by combining a single value with a range that is narrowed by one value.
* Tail-Call Optimization (TCO):
    * When the recursion happens at the very end of the function, it is considered *tail recursion*.
    * In some functional programming languages, compilers perform TCO to automatically transform recursive functions into efficient, low-overhead loops.
* Python's limitations with recursion:
    * Python lacks a compiler to perform TCO; it sacrifices performance and memory for clarity.
    * Python enforces a strict default recursion limit of 1,000 to detect improperly defined base cases, which restricts deep recursion.
* Manual optimization: 
    * Because Python does not optimize recursion natively, developers are forced to perform manual optimizations, such as optimizing a purely recursive function to use an explicit `for` loop in a generator expression.

In [17]:
%%ipytest


def flatten_recursive(nested: list[Any]) -> list[Any]:
    """Flatten a nested list.

    Useful for hierarchical JSON data, nested directory trees, etc.
    """
    # Base case: if the list is empty, return an empty list
    if not nested:
        return []

    first, *rest = nested

    # Non-empty case: decompose the problem into a smaller piece (`first`) and a
    # narrowed subproblem (`rest`), combining them recursively.
    if isinstance(first, list):
        return flatten_recursive(first) + flatten_recursive(rest)

    return [first, *flatten_recursive(rest)]


def flatten_generator(nested: list[Any]) -> Generator[Any]:
    # Use a stack to track items to process without blowing the call stack
    stack = list(reversed(nested))
    while stack:
        current = stack.pop()
        if isinstance(current, list):
            # Push items back onto the stack in reverse order to preserve sequence
            stack.extend(reversed(current))
        else:
            yield current


@pytest.mark.parametrize("flatten_func", [flatten_recursive, flatten_generator])
@pytest.mark.parametrize(
    ("nested_input", "expected_output"),
    [
        ([[1, 2], [3, [4, 5]]], [1, 2, 3, 4, 5]),
        ([["a", ["b"]], "c"], ["a", "b", "c"]),
    ],
)
def test_flatten(
    flatten_func, nested_input: list[Any], expected_output: list[Any],
) -> None:
    # Convert the result to a list to support both regular lists and generators
    assert list(flatten_func(nested_input)) == expected_output

....                                                                                         [100%]
4 passed in 0.02s


## Functional type systems

* Some functional programming languages, such as Haskell and Scala, are statically compiled, and depend on declared types for functions and their arguments.
* These languages have sophisticated type-matching rules so that a generic function can be written, which works for a variety of related types.
* Python doesn't need this because it's a dynamic language.